# 00E — Reproduce access errors with the current role

**No required settings. Run the cells in order with your permissions administrator.**
Every AWS command is directly in its cell. On failure, the cell prints the original
exception and you can continue. There are no reporting helpers or imported application code.

The notebook attempts to create, update and start its own uniquely named pipeline.
The start cell launches **one small, billable Processing job** if permitted.
Role and image are selected automatically. The S3 checks only read:
enter a bucket in the first cell, or leave it blank to use the first listed bucket.

Only boto3 is needed. Nothing changes IAM permissions or existing pipelines.
Use a fresh kernel for a new probe. Keep outputs in your environment and clear them
before sharing this notebook. **00D remains available for more extensive checks.**

## 1. Start with the current AWS credentials

- Leave `bucket_name` blank, or enter a bucket you want to check.
- Use the current region and create AWS clients.
- Choose a unique name for this notebook's pipeline.

In [ ]:
import json
import uuid
from pprint import pprint

import boto3
from botocore.config import Config

bucket_name = ""  # Optional. Empty means use the first bucket returned by ListBuckets.

session = boto3.Session()
region = session.region_name or "us-east-1"
config = Config(connect_timeout=5, read_timeout=20, retries={"max_attempts": 0})
sts = session.client("sts", region_name=region, config=config)
iam = session.client("iam", region_name=region, config=config)
sm = session.client("sagemaker", region_name=region, config=config)
s3 = session.client("s3", region_name=region, config=config)
logs = session.client("logs", region_name=region, config=config)

probe_id = uuid.uuid4().hex
pipeline_name = "managed-access-" + probe_id[:10]
role_name = role_arn = None
pipeline_arn = execution_arn = job_name = object_key = None
print("Region:", region, "(fallback us-east-1)" if not session.region_name else "")
print("New pipeline:", pipeline_name)

## 2. Get the current role from STS

- Call GetCallerIdentity with the notebook's credentials.
- Derive an IAM role ARN from the current assumed-role identity.
- Print that ARN; its IAM path is unverified until the next check succeeds.

In [ ]:
try:
    caller = sts.get_caller_identity()
    pprint(caller)
    if ":assumed-role/" in caller["Arn"]:
        role_name = caller["Arn"].split(":assumed-role/", 1)[1].split("/")[0]
        partition = caller["Arn"].split(":")[1]
        role_arn = f"arn:{partition}:iam::{caller['Account']}:role/{role_name}"
        print("Role from STS (IAM path unverified):", role_arn)
    else:
        print("The caller is not an assumed role; no SageMaker role can be inferred.")
except Exception as error:
    print(error)

## 3. Try to resolve the exact IAM role ARN

- Call GetRole for the name discovered above.
- Use its exact ARN, including any IAM path, if allowed.
- If this fails, still attempt CreatePipeline with the STS-derived ARN.

In [ ]:
if role_name:
    try:
        response = iam.get_role(RoleName=role_name)
        role_arn = response["Role"]["Arn"]
        print("Exact role:", role_arn)
    except Exception as error:
        print(error)
        print("Will still try the STS-derived ARN:", role_arn)
else:
    print("No role name was obtained from STS.")

## 4. Select the standard AWS image automatically

- Look up the AWS scikit-learn image registry for the current region.
- Build the image URI without needing Docker or the SageMaker SDK.
- Print the exact image that the CreatePipeline cell will use.

This is AWS's published registry map from the
[SageMaker SDK image configuration](https://github.com/aws/sagemaker-python-sdk/blob/v2.253.1/src/sagemaker/image_uri_config/sklearn.json).
No image name needs to be entered for the listed regions.

In [ ]:
aws_image_registries = {
    "af-south-1": "510948584623",
    "ap-east-1": "651117190479",
    "ap-northeast-1": "354813040037",
    "ap-northeast-2": "366743142698",
    "ap-northeast-3": "867004704886",
    "ap-south-1": "720646828776",
    "ap-south-2": "628508329040",
    "ap-southeast-1": "121021644041",
    "ap-southeast-2": "783357654285",
    "ap-southeast-3": "951798379941",
    "ap-southeast-4": "106583098589",
    "ca-central-1": "341280168497",
    "ca-west-1": "190319476487",
    "eu-central-1": "492215442770",
    "eu-central-2": "680994064768",
    "eu-north-1": "662702820516",
    "eu-south-1": "978288397137",
    "eu-south-2": "104374241257",
    "eu-west-1": "141502667606",
    "eu-west-2": "764974769150",
    "eu-west-3": "659782779980",
    "il-central-1": "898809789911",
    "me-central-1": "272398656194",
    "me-south-1": "801668240914",
    "sa-east-1": "737474898029",
    "us-east-1": "683313688378",
    "us-east-2": "257758044811",
    "us-west-1": "746614075791",
    "us-west-2": "246618743249"
}
registry = aws_image_registries.get(region)
image_uri = (
    f"{registry}.dkr.ecr.{region}.amazonaws.com/sagemaker-scikit-learn:1.2-1-cpu-py3"
    if registry else None
)
print("Image:", image_uri or "No standard image is listed for this region.")

## 5. CreatePipeline — the main permission check

- Send the current role, selected image and the one-step definition shown in this cell.
- Print the exact request values and the CreatePipeline response or original exception.
- Attempt the call even if GetRole failed; there is no separate-role requirement or enable switch.

SageMaker requires a role ARN and pipeline definition in this API request. The notebook
supplies them automatically. The **same current role** is used for the pipeline and job
solely to test the access you have today; no IAM policies are changed. If a role path,
image or other platform control prevents access, keep that error for the administrator.

In [ ]:
# This is the entire pipeline: one Processing job that prints 10 + 20.
definition = {
    "Version": "2020-12-01",
    "Steps": [{
        "Name": "CheckAccess",
        "Type": "Processing",
        "Arguments": {
            "ProcessingJobName": pipeline_name + "-job",
            "RoleArn": role_arn,
            "AppSpecification": {
                "ImageUri": image_uri,
                "ContainerEntrypoint": [
                    "python3", "-c", "print('ACCESS_CHECK_OK:', 10 + 20, flush=True)"
                ],
            },
            "ProcessingResources": {"ClusterConfig": {
                "InstanceCount": 1, "InstanceType": "ml.m5.large", "VolumeSizeInGB": 10
            }},
            "NetworkConfig": {"EnableNetworkIsolation": True},
            "StoppingCondition": {"MaxRuntimeInSeconds": 120},
        },
    }],
}
print("PipelineName:", pipeline_name)
print("RoleArn:", role_arn)
print("ImageUri:", image_uri)

try:
    response = sm.create_pipeline(
        PipelineName=pipeline_name,
        RoleArn=role_arn,
        PipelineDefinition=json.dumps(definition),
        PipelineDescription="Current-role access check",
        ClientRequestToken=probe_id,
    )
    pipeline_arn = response["PipelineArn"]
    pprint(response)
except Exception as error:
    print(error)

## 6. UpdatePipeline

- Update only the new pipeline whose creation succeeded above.
- Pass the same current role and change its description.
- Print the response or original error; an update denial does not prevent testing start.

In [ ]:
if pipeline_arn:
    try:
        response = sm.update_pipeline(
            PipelineName=pipeline_name,
            RoleArn=role_arn,
            PipelineDescription="Updated current-role access check",
        )
        pprint(response)
    except Exception as error:
        print(error)
else:
    print("Not tested: CreatePipeline did not succeed.")

## 7. StartPipelineExecution

- Start this notebook's pipeline, even if UpdatePipeline failed.
- Launch one small, billable Processing job if permitted.
- Reuse this kernel's request token so retrying this cell does not start another execution.

In [ ]:
if pipeline_arn:
    try:
        response = sm.start_pipeline_execution(
            PipelineName=pipeline_name,
            PipelineExecutionDisplayName="access-" + probe_id[:10],
            ClientRequestToken=probe_id,
        )
        execution_arn = response["PipelineExecutionArn"]
        pprint(response)
    except Exception as error:
        print(error)
else:
    print("Not tested: CreatePipeline did not succeed.")

## 8. List buckets

- Call ListBuckets independently of all pipeline checks.
- If no input bucket was supplied, choose the first bucket returned.
- Preserve a supplied bucket even if listing is denied, so its direct checks can still run.

In [ ]:
try:
    response = s3.list_buckets(MaxBuckets=5)
    pprint(response)
    if not bucket_name and response.get("Buckets"):
        bucket_name = response["Buckets"][0]["Name"]
except Exception as error:
    print(error)
print("Bucket for the next checks:", bucket_name or "none discovered")

## 9. List files in that bucket

- Call ListObjectsV2 for the selected bucket.
- Look at the first ten objects and choose the first nonempty file.
- Print the selected key; no file path needs to be entered.

In [ ]:
if bucket_name:
    try:
        response = s3.list_objects_v2(Bucket=bucket_name, MaxKeys=10)
        pprint(response)
        object_key = next(
            (item["Key"] for item in response.get("Contents", [])
             if item.get("Size", 0) > 0 and not item["Key"].endswith("/")),
            None,
        )
        print("Selected file:", object_key or "no nonempty file in this page")
    except Exception as error:
        print(error)
else:
    print("Not tested: no bucket was supplied or discovered.")

## 10. Read that file

- Call GetObject for the automatically selected file.
- Read up to 1 KiB and close the stream without downloading the whole file.
- Show the byte count and metadata; do not print file contents.

In [ ]:
if bucket_name and object_key:
    print("Object:", f"s3://{bucket_name}/{object_key}")
    try:
        response = s3.get_object(Bucket=bucket_name, Key=object_key)
        with response["Body"] as body:
            bytes_read = len(body.read(1024))
        pprint({
            "bytes_read": bytes_read,
            "object_size": response.get("ContentLength"),
            "version_id": response.get("VersionId"),
            "etag": response.get("ETag"),
        })
    except Exception as error:
        print(error)
else:
    print("Not tested: no file was discovered.")

## 11. DescribePipelineExecution

- Check the actual execution status after starting the pipeline.
- Read FailureReason if the pipeline cannot create or run its job.
- Rerun this cell if it says Executing; starting is asynchronous and may take a few minutes.

In [ ]:
if execution_arn:
    try:
        response = sm.describe_pipeline_execution(PipelineExecutionArn=execution_arn)
        pprint(response)
    except Exception as error:
        print(error)
else:
    print("Not tested: StartPipelineExecution did not succeed.")

## 12. ListPipelineExecutionSteps

- Inspect the step even if the execution describe call was denied.
- Show any job-creation failure reported by the pipeline.
- Pick up the Processing job name automatically if a job exists.

In [ ]:
if execution_arn:
    try:
        response = sm.list_pipeline_execution_steps(PipelineExecutionArn=execution_arn)
        pprint(response)
        for step in response.get("PipelineExecutionSteps", []):
            job_arn = step.get("Metadata", {}).get("ProcessingJob", {}).get("Arn")
            if job_arn:
                job_name = job_arn.rsplit("/", 1)[-1]
        print("Processing job:", job_name or "not created yet")
    except Exception as error:
        print(error)
else:
    print("Not tested: no execution was started.")

## 13. DescribeProcessingJob

- Read the job's status and runtime role.
- Show FailureReason if processing failed.
- Rerun after the previous cell discovers the job name.

In [ ]:
if job_name:
    try:
        response = sm.describe_processing_job(ProcessingJobName=job_name)
        pprint(response)
    except Exception as error:
        print(error)
else:
    print("Not tested: no job name was returned yet.")

## 14. Read the job's CloudWatch output

- Call FilterLogEvents for the job discovered above.
- Look for `ACCESS_CHECK_OK: 30` in the messages.
- Rerun after completion if logs have not arrived yet.

In [ ]:
if job_name:
    try:
        response = logs.filter_log_events(
            logGroupName="/aws/sagemaker/ProcessingJobs",
            logStreamNamePrefix=job_name + "/",
            limit=20,
        )
        pprint(response)
    except Exception as error:
        print(error)
else:
    print("Not tested: no job name was returned yet.")

## 15. Optional: delete this notebook's pipeline

- Leave the switch false to keep the pipeline for inspection.
- After the execution finishes, set it true to delete only the pipeline created in this kernel.
- Keep job history and logs; no S3 objects are deleted.

In [ ]:
DELETE_PIPELINE = False

if DELETE_PIPELINE and pipeline_arn:
    print("Deleting this probe:", pipeline_arn)
    try:
        response = sm.delete_pipeline(
            PipelineName=pipeline_name,
            ClientRequestToken=probe_id,
        )
        pprint(response)
        pipeline_arn = None
    except Exception as error:
        print(error)
else:
    print("Cleanup is off, or this notebook did not create a pipeline.")

Creating the pipeline, starting it, completing the Processing job and reading its logs
are separate outcomes. Keep the exact errors from whichever step fails.
This probe does not execute Training or Batch Transform, register models, write to S3,
push images or deploy infrastructure. Production network, image and role settings
remain separate from this small diagnostic.